# Lab 2 Phase 1: ResNet Super-Resolution for Mobilint NPU

Same-resolution image restoration (256x256 -> 256x256) using a ResNet architecture
designed for Mobilint NPU compatibility.

**NPU-native ops used:** Conv2d, BatchNorm2d, PReLU
**Only CPU-fallback op:** element-wise Add (skip connections)

**Kernel:** Select "Python 3.9 (torch)" as the Jupyter kernel before running.

In [7]:
from pathlib import Path
import json
import math
import os
import random
import time
import warnings

warnings.filterwarnings("ignore", message=".*legacy TorchScript-based ONNX.*")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from torchvision import transforms

try:
    import onnx
except ImportError:
    onnx = None

try:
    import onnxruntime as ort
except ImportError:
    ort = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

Using device: cpu


## 1. Data Loading

Recursively collects all `.png` files from the nested subdirectories under
`Data/HR_train/` and `Data/LR_train/`, pairs them by filename stem.
Validation images come from `Data/val/` and are **never** used for training.

In [8]:
NOTEBOOK_DIR = Path(os.path.abspath("")).resolve()
if (NOTEBOOK_DIR / "lab2_phase1_resnet.ipynb").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "Lab 2 Phase 1").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
else:
    PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_ROOT = PROJECT_ROOT / "Data"
assert DATA_ROOT.exists(), f"Data directory not found at {DATA_ROOT}"
OUTPUT_DIR = NOTEBOOK_DIR / "runs" / "resnet_sr" if (NOTEBOOK_DIR / "lab2_phase1_resnet.ipynb").exists() else Path("runs/resnet_sr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data root: {DATA_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")

HR_TRAIN_ROOT = DATA_ROOT / "HR_train"
LR_TRAIN_ROOT = DATA_ROOT / "LR_train"
HR_VAL_DIR = DATA_ROOT / "val" / "HR_val"
LR_VAL_DIR = DATA_ROOT / "val" / "LR_val"


def collect_paired_by_subfolder(lr_root: Path, hr_root: Path) -> list[tuple[Path, Path, str]]:
    """Pair LR/HR images across matching numbered subdirectories.

    Subdirectories are matched by numeric suffix: HR_trainN <-> LR_trainN.
    Images are paired by filename stem within each subfolder pair.
    This handles duplicate stems across different subfolders correctly.
    """
    hr_subdirs = sorted([d for d in hr_root.iterdir() if d.is_dir()])
    pairs = []
    for hr_sub in hr_subdirs:
        suffix = hr_sub.name.replace("HR_train", "")
        lr_sub = lr_root / f"LR_train{suffix}"
        if not lr_sub.exists():
            print(f"  WARNING: no matching LR subfolder for {hr_sub.name}")
            continue
        hr_imgs = {p.stem: p for p in sorted(hr_sub.glob("*.png"))}
        lr_imgs = {p.stem: p for p in sorted(lr_sub.glob("*.png"))}
        common = sorted(set(hr_imgs) & set(lr_imgs))
        for stem in common:
            label = f"{hr_sub.name}/{stem}"
            pairs.append((lr_imgs[stem], hr_imgs[stem], label))
        unmatched_hr = len(set(hr_imgs) - set(lr_imgs))
        unmatched_lr = len(set(lr_imgs) - set(hr_imgs))
        if unmatched_hr or unmatched_lr:
            print(f"  {hr_sub.name}: {unmatched_hr} HR-only, {unmatched_lr} LR-only")
        print(f"  {hr_sub.name} <-> LR_train{suffix}: {len(common)} pairs")
    return pairs


def collect_paired_flat(lr_dir: Path, hr_dir: Path) -> list[tuple[Path, Path, str]]:
    """Pair images in two flat directories by filename stem."""
    hr_imgs = {p.stem: p for p in sorted(hr_dir.glob("*.png"))}
    lr_imgs = {p.stem: p for p in sorted(lr_dir.glob("*.png"))}
    common = sorted(set(hr_imgs) & set(lr_imgs))
    return [(lr_imgs[s], hr_imgs[s], s) for s in common]


print("=== Training data ===")
train_pairs = collect_paired_by_subfolder(LR_TRAIN_ROOT, HR_TRAIN_ROOT)
print(f"Total training pairs: {len(train_pairs)}\n")

print("=== Validation data ===")
val_pairs = collect_paired_flat(LR_VAL_DIR, HR_VAL_DIR)
print(f"Total validation pairs: {len(val_pairs)}")

Data root: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Data
Output dir: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 1/runs/resnet_sr
=== Training data ===
  HR_train1 <-> LR_train1: 663 pairs
  HR_train2 <-> LR_train2: 782 pairs
  HR_train3 <-> LR_train3: 810 pairs
  HR_train4 <-> LR_train4: 781 pairs
Total training pairs: 3036

=== Validation data ===
Total validation pairs: 100


In [9]:
class PairedSRDataset(Dataset):
    """Paired LR/HR dataset with synchronized augmentation for training."""

    def __init__(self, pairs: list[tuple[Path, Path, str]], train: bool = True):
        self.pairs = pairs
        self.train = train
        self.to_tensor = transforms.ToTensor()  # HWC uint8 [0,255] -> CHW float [0,1]

    def __len__(self):
        return len(self.pairs)

    def _load(self, path: Path) -> Image.Image:
        img = Image.open(path).convert("RGB")
        assert img.size == (256, 256), f"Expected 256x256, got {img.size} for {path}"
        return img

    def __getitem__(self, idx):
        lr_path, hr_path, stem = self.pairs[idx]
        lr_img = self._load(lr_path)
        hr_img = self._load(hr_path)

        if self.train:
            if random.random() > 0.5:
                lr_img = lr_img.transpose(Image.FLIP_LEFT_RIGHT)
                hr_img = hr_img.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                lr_img = lr_img.transpose(Image.FLIP_TOP_BOTTOM)
                hr_img = hr_img.transpose(Image.FLIP_TOP_BOTTOM)
            k = random.randint(0, 3)
            if k > 0:
                rot = {1: Image.ROTATE_90, 2: Image.ROTATE_180, 3: Image.ROTATE_270}[k]
                lr_img = lr_img.transpose(rot)
                hr_img = hr_img.transpose(rot)

        return self.to_tensor(lr_img), self.to_tensor(hr_img), stem


BATCH_SIZE = 8
NUM_WORKERS = 0  # must be 0 for notebook (Dataset class can't be pickled by spawn workers)
SEED = 255

train_dataset = PairedSRDataset(train_pairs, train=True)
val_dataset = PairedSRDataset(val_pairs, train=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
)

lr_batch, hr_batch, names = next(iter(train_loader))
print(f"Train dataset:  {len(train_dataset)} pairs")
print(f"Val dataset:    {len(val_dataset)} pairs")
print(f"LR batch shape: {tuple(lr_batch.shape)}, dtype={lr_batch.dtype}, range=[{lr_batch.min():.2f}, {lr_batch.max():.2f}]")
print(f"HR batch shape: {tuple(hr_batch.shape)}, dtype={hr_batch.dtype}, range=[{hr_batch.min():.2f}, {hr_batch.max():.2f}]")
print(f"Sample stems:   {list(names[:3])}")

Train dataset:  3036 pairs
Val dataset:    100 pairs
LR batch shape: (8, 3, 256, 256), dtype=torch.float32, range=[0.00, 1.00]
HR batch shape: (8, 3, 256, 256), dtype=torch.float32, range=[0.00, 1.00]
Sample stems:   ['HR_train4/000382', 'HR_train1/000422', 'HR_train1/000458']


## 2. Model Definition: ResNet-SR

Global residual learning: the model predicts `delta = HR - LR`, output = `LR + delta`.

All heavy ops (Conv, BN, PReLU) are Mobilint NPU-native. Only element-wise Add
(skip connections) falls back to CPU. No failed ops.

In [10]:
class ResBlock(nn.Module):
    """Pre-activation-style residual block using only NPU-native ops.
    Conv2d (Convolution: NPU native)
    BatchNorm2d (Batchnorm: NPU native)
    PReLU (PRelu: NPU native)
    + skip (Adding: CPU Fallback)
    """
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.act = nn.PReLU(num_parameters=channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return identity + out


class ResNetSR(nn.Module):
    """ResNet-based same-resolution super-resolution model.

    Architecture: Stem -> N x ResBlock -> Tail + global skip
    Input/Output: [B, 3, 256, 256]
    """
    def __init__(self, num_blocks: int = 10, channels: int = 48):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.PReLU(num_parameters=channels),
        )
        self.body = nn.Sequential(*[ResBlock(channels) for _ in range(num_blocks)])
        self.tail = nn.Conv2d(channels, 3, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x
        feat = self.stem(x)
        feat = self.body(feat)
        delta = self.tail(feat)
        return identity + delta


NPU_NATIVE_TYPES = (nn.Conv2d, nn.BatchNorm2d, nn.PReLU)
FORBIDDEN_TYPES = (
    nn.ReLU, nn.Sigmoid, nn.Softmax, nn.Upsample,
    nn.LayerNorm, nn.GroupNorm,
)

def assert_npu_compatible(model: nn.Module) -> None:
    """Verify no forbidden ops are in the model graph."""
    for name, module in model.named_modules():
        if isinstance(module, FORBIDDEN_TYPES):
            raise TypeError(f"Forbidden NPU op in '{name}': {module.__class__.__name__}")

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


NUM_BLOCKS = 10
CHANNELS = 48

model = ResNetSR(num_blocks=NUM_BLOCKS, channels=CHANNELS).to(device)
assert_npu_compatible(model)

with torch.no_grad():
    dummy = torch.randn(1, 3, 256, 256, device=device)
    out = model(dummy)

print(f"Model: ResNetSR(blocks={NUM_BLOCKS}, channels={CHANNELS})")
print(f"Parameters: {count_parameters(model):,}")
print(f"Input shape:  {tuple(dummy.shape)}")
print(f"Output shape: {tuple(out.shape)}")

npu_ops = {"Convolution": 0, "Batchnorm": 0, "PReLU": 0}
cpu_fallback = 0
for m in model.modules():
    if isinstance(m, nn.Conv2d):
        npu_ops["Convolution"] += 1
    elif isinstance(m, nn.BatchNorm2d):
        npu_ops["Batchnorm"] += 1
    elif isinstance(m, nn.PReLU):
        npu_ops["PReLU"] += 1

cpu_fallback = NUM_BLOCKS + 1  # block skips + global skip
print(f"\nNPU-native ops: {npu_ops}")
print(f"CPU Fallback (Adding): {cpu_fallback} instances")
print(f"Failed ops: 0")

Model: ResNetSR(blocks=10, channels=48)
Parameters: 419,859
Input shape:  (1, 3, 256, 256)
Output shape: (1, 3, 256, 256)

NPU-native ops: {'Convolution': 22, 'Batchnorm': 21, 'PReLU': 11}
CPU Fallback (Adding): 11 instances
Failed ops: 0


## 3. Training Infrastructure

Charbonnier loss, cosine LR schedule with warmup, PSNR metric, checkpoint management.

In [11]:
TRAIN_CFG = {
    "epochs": 100,
    "batch_size": BATCH_SIZE,
    "lr": 3e-4,
    "weight_decay": 1e-4,
    "warmup_epochs": 5,
    "min_lr_ratio": 0.05,
    "charbonnier_eps": 1e-6,
    "seed": SEED,
}

RESUME_TRAINING = True   # True = resume from last.pt if it exists; False = train from scratch

print(f"Config: {TRAIN_CFG}")
print(f"Resume: {RESUME_TRAINING}")


def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def charbonnier_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Smooth L1 variant: sqrt((pred-target)^2 + eps^2). Optimizes PSNR well."""
    diff = pred - target
    return torch.mean(torch.sqrt(diff * diff + eps * eps))


def compute_psnr(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    """Per-image PSNR on clamped [0,1] predictions."""
    pred = pred.clamp(0.0, 1.0)
    target = target.clamp(0.0, 1.0)
    mse = torch.mean((pred - target) ** 2, dim=(1, 2, 3)).clamp_min(1e-12)
    return -10.0 * torch.log10(mse)


def lr_for_epoch(epoch: int, total: int, base_lr: float, warmup: int, min_ratio: float) -> float:
    if warmup > 0 and epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, total - warmup - 1)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    min_lr = base_lr * min_ratio
    return min_lr + (base_lr - min_lr) * cosine


def train_one_epoch(model, loader, optimizer, cfg):
    model.train()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for lr_img, hr_img, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        pred = model(lr_img)
        loss = charbonnier_loss(pred, hr_img, eps=cfg["charbonnier_eps"])
        loss.backward()
        optimizer.step()
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        with torch.no_grad():
            total_psnr += compute_psnr(pred.detach(), hr_img).sum().item()
        n += bs
    return {"train_loss": total_loss / max(1, n), "train_psnr": total_psnr / max(1, n)}


@torch.no_grad()
def validate(model, loader, cfg):
    model.eval()
    total_loss, total_psnr, n = 0.0, 0.0, 0
    for lr_img, hr_img, _ in loader:
        lr_img = lr_img.to(device, non_blocking=True)
        hr_img = hr_img.to(device, non_blocking=True)
        pred = model(lr_img)
        loss = charbonnier_loss(pred, hr_img, eps=cfg["charbonnier_eps"])
        psnr = compute_psnr(pred, hr_img)
        bs = lr_img.size(0)
        total_loss += loss.item() * bs
        total_psnr += psnr.sum().item()
        n += bs
    return {"val_loss": total_loss / max(1, n), "val_psnr": total_psnr / max(1, n)}


def save_checkpoint(path, model, optimizer, epoch, metrics, best_psnr):
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": metrics,
        "best_psnr": best_psnr,
        "train_cfg": TRAIN_CFG,
    }, path)


def load_history(metrics_path: Path) -> list[dict]:
    """Reload training history from metrics.jsonl."""
    history = []
    if metrics_path.exists():
        for line in metrics_path.read_text().strip().splitlines():
            if line:
                history.append(json.loads(line))
    return history


def fit(model, train_loader, val_loader, output_dir, cfg=TRAIN_CFG, resume=True):
    """Train with checkpoint resume support.

    Args:
        resume: If True and last.pt exists, resume from that checkpoint.
                If False, always start fresh (overwrites existing checkpoints).
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "metrics.jsonl"
    last_ckpt_path = output_dir / "last.pt"
    best_ckpt_path = output_dir / "best.pt"

    optimizer = AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    start_epoch = 0
    best_psnr = float("-inf")
    history = []
    cumulative_time = 0.0

    if resume and last_ckpt_path.exists():
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"]
        best_psnr = ckpt.get("best_psnr", float("-inf"))
        history = load_history(metrics_path)
        cumulative_time = sum(r.get("seconds", 0) for r in history)
        print(f"RESUMING from epoch {start_epoch}/{cfg['epochs']}")
        print(f"  Best PSNR so far: {best_psnr:.3f} dB")
        print(f"  Prior training time: {cumulative_time/60:.1f} min")
        print(f"  Remaining epochs: {cfg['epochs'] - start_epoch}")
    else:
        set_seed(cfg["seed"])
        metrics_path.write_text("")
        if not resume and last_ckpt_path.exists():
            print("FRESH START (ignoring existing checkpoints)")
        else:
            print("FRESH START (no existing checkpoints)")

    if start_epoch >= cfg["epochs"]:
        print(f"\nAlready trained {start_epoch} epochs (target: {cfg['epochs']}). "
              f"Increase TRAIN_CFG['epochs'] to train more.")
        return history

    print(f"\n{'epoch':>5} | {'lr':>8} | {'train_loss':>10} {'train_psnr':>10} | "
          f"{'val_loss':>8} {'val_psnr':>8} | {'best':>8} | {'time':>5} {'total':>7}")
    print("-" * 95)

    for epoch in range(start_epoch, cfg["epochs"]):
        epoch_lr = lr_for_epoch(epoch, cfg["epochs"], cfg["lr"], cfg["warmup_epochs"], cfg["min_lr_ratio"])
        for pg in optimizer.param_groups:
            pg["lr"] = epoch_lr

        t0 = time.time()
        train_metrics = train_one_epoch(model, train_loader, optimizer, cfg)
        val_metrics = validate(model, val_loader, cfg)
        elapsed = time.time() - t0
        cumulative_time += elapsed

        row = {
            "epoch": epoch + 1,
            "lr": epoch_lr,
            "train_loss": train_metrics["train_loss"],
            "train_psnr": train_metrics["train_psnr"],
            "val_loss": val_metrics["val_loss"],
            "val_psnr": val_metrics["val_psnr"],
            "seconds": round(elapsed, 1),
        }
        history.append(row)
        with metrics_path.open("a") as f:
            f.write(json.dumps(row) + "\n")

        is_best = row["val_psnr"] > best_psnr
        if is_best:
            best_psnr = row["val_psnr"]

        save_checkpoint(last_ckpt_path, model, optimizer, epoch + 1, row, best_psnr)
        if is_best:
            save_checkpoint(best_ckpt_path, model, optimizer, epoch + 1, row, best_psnr)

        marker = " *" if is_best else ""
        print(
            f"{epoch+1:3d}/{cfg['epochs']:3d} | "
            f"{epoch_lr:.2e} | "
            f"{row['train_loss']:10.6f} {row['train_psnr']:8.3f} dB | "
            f"{row['val_loss']:8.6f} {row['val_psnr']:6.3f} dB | "
            f"{best_psnr:6.3f} dB | "
            f"{elapsed:5.1f}s {cumulative_time/60:5.1f}m{marker}"
        )

    best_row = max(history, key=lambda r: r["val_psnr"])
    print(f"\nTraining complete. Best: epoch {best_row['epoch']}, val_psnr={best_row['val_psnr']:.3f} dB")
    print(f"Total training time: {cumulative_time/60:.1f} min")
    print(f"Checkpoints: {output_dir}")
    return history

Config: {'epochs': 100, 'batch_size': 8, 'lr': 0.0003, 'weight_decay': 0.0001, 'warmup_epochs': 5, 'min_lr_ratio': 0.05, 'charbonnier_eps': 1e-06, 'seed': 255}
Resume: True


## 4. Smoke Test

Quick 3-epoch run on 16 images to verify the full pipeline works before committing
to a long training run.

In [12]:
RUN_SMOKE_TEST = True

if RUN_SMOKE_TEST:
    smoke_cfg = {**TRAIN_CFG, "epochs": 3}
    smoke_train = PairedSRDataset(train_pairs[:16], train=True)
    smoke_val = PairedSRDataset(val_pairs[:8], train=False)
    smoke_tl = DataLoader(smoke_train, batch_size=4, shuffle=True, num_workers=0)
    smoke_vl = DataLoader(smoke_val, batch_size=4, shuffle=False, num_workers=0)

    smoke_model = ResNetSR(num_blocks=NUM_BLOCKS, channels=CHANNELS).to(device)
    smoke_opt = AdamW(smoke_model.parameters(), lr=smoke_cfg["lr"], weight_decay=smoke_cfg["weight_decay"])

    for ep in range(smoke_cfg["epochs"]):
        tm = train_one_epoch(smoke_model, smoke_tl, smoke_opt, smoke_cfg)
        vm = validate(smoke_model, smoke_vl, smoke_cfg)
        print(f"  smoke epoch {ep+1}: train_loss={tm['train_loss']:.6f} "
              f"train_psnr={tm['train_psnr']:.3f} dB | val_psnr={vm['val_psnr']:.3f} dB")

    del smoke_model, smoke_opt, smoke_tl, smoke_vl
    if device.type == "cuda":
        torch.cuda.empty_cache()
    print("Smoke test passed.")

  smoke epoch 1: train_loss=1.208441 train_psnr=6.277 dB | val_psnr=18.015 dB
  smoke epoch 2: train_loss=0.636315 train_psnr=7.483 dB | val_psnr=19.012 dB
  smoke epoch 3: train_loss=0.515136 train_psnr=9.151 dB | val_psnr=17.854 dB
Smoke test passed.


## 5. Full Training

Set `RESUME_TRAINING` in section 3 above to control behavior:
- `True` = resume from `last.pt` if it exists (crash recovery, or extend training)
- `False` = train from scratch (overwrites existing checkpoints)

To train more epochs, increase `TRAIN_CFG["epochs"]` in section 3 and re-run.
A `*` marker on the epoch line indicates a new best validation PSNR.

In [13]:
model = ResNetSR(num_blocks=NUM_BLOCKS, channels=CHANNELS).to(device)
assert_npu_compatible(model)
print(f"Target: {TRAIN_CFG['epochs']} epochs | {len(train_dataset)} train / {len(val_dataset)} val")
print(f"Resume: {RESUME_TRAINING} | Checkpoints: {OUTPUT_DIR}\n")

history = fit(model, train_loader, val_loader, OUTPUT_DIR, resume=RESUME_TRAINING)

Target: 100 epochs | 3036 train / 100 val
Resume: True | Checkpoints: /Users/cyrilgoud/Desktop/repos/personal/Lab 2/Lab 2 Phase 1/runs/resnet_sr

RESUMING from epoch 10/100
  Best PSNR so far: 21.321 dB
  Prior training time: 120.5 min
  Remaining epochs: 90

epoch |       lr | train_loss train_psnr | val_loss val_psnr |     best |  time   total
-----------------------------------------------------------------------------------------------
 11/100 | 2.98e-04 |   0.042810   25.858 dB | 0.066275 21.099 dB | 21.321 dB | 737.1s 132.8m
 12/100 | 2.97e-04 |   0.041857   26.046 dB | 0.062418 21.362 dB | 21.362 dB | 739.9s 145.2m *


KeyboardInterrupt: 

## 6. ONNX Export

Export best checkpoint to ONNX with opset 13, fixed shape `[1, 3, 256, 256]`,
named I/O tensors `input`/`output`. Constant folding fuses BatchNorm into Conv.
Optionally verify PyTorch vs ONNX Runtime parity.

In [ ]:
def load_checkpoint(model, checkpoint_path, map_location="cpu"):
    ckpt = torch.load(checkpoint_path, map_location=map_location)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"])
    else:
        model.load_state_dict(ckpt)
    return ckpt


def export_to_onnx(checkpoint_path, onnx_path, verify=False, sample_lr_path=None):
    checkpoint_path = Path(checkpoint_path)
    onnx_path = Path(onnx_path)
    onnx_path.parent.mkdir(parents=True, exist_ok=True)

    export_model = ResNetSR(num_blocks=NUM_BLOCKS, channels=CHANNELS).to(device)
    ckpt = load_checkpoint(export_model, checkpoint_path, map_location=device)
    export_model.eval()

    if "metrics" in ckpt:
        print(f"Checkpoint epoch {ckpt['metrics'].get('epoch', '?')}, "
              f"val_psnr={ckpt['metrics'].get('val_psnr', '?'):.3f} dB")

    dummy = torch.randn(1, 3, 256, 256, device=device)
    export_kwargs = dict(
        export_params=True,
        opset_version=13,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
    )

    try:
        torch.onnx.export(export_model, dummy, str(onnx_path), dynamo=False, **export_kwargs)
    except TypeError:
        torch.onnx.export(export_model, dummy, str(onnx_path), **export_kwargs)

    print(f"Exported to {onnx_path} ({onnx_path.stat().st_size / 1024:.0f} KB)")

    if onnx is not None:
        onnx_model = onnx.load(str(onnx_path))
        onnx.checker.check_model(onnx_model)
        print("ONNX checker: passed")
    else:
        print("onnx package not installed, skipped checker")

    if verify and ort is not None:
        if sample_lr_path is None:
            raise ValueError("sample_lr_path required for verification")
        sample_img = Image.open(sample_lr_path).convert("RGB")
        sample_t = transforms.ToTensor()(sample_img).unsqueeze(0).to(device)

        with torch.no_grad():
            torch_out = export_model(sample_t).cpu().numpy()

        sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
        ort_out = sess.run(["output"], {"input": sample_t.cpu().numpy()})[0]
        diff = abs(torch_out - ort_out)
        print(f"Parity check: max_diff={diff.max():.8f}, mean_diff={diff.mean():.8f}")

    return onnx_path

In [ ]:
sample_lr = sorted(LR_VAL_DIR.glob("*.png"))[0]
onnx_path = export_to_onnx(
    OUTPUT_DIR / "best.pt",
    OUTPUT_DIR / "best.onnx",
    verify=True,
    sample_lr_path=sample_lr,
)
print(f"\nReady for Mobilint compilation: {onnx_path}")